In [ ]:
# lseg.data取得 -> weekly_intensity 出力まで（pipeline抜粋）

from pathlib import Path
import sys
import pandas as pd
import lseg.data as ld

PROJECT_ROOT = Path("/Users/kencharoff/workspace/projects/thematic-topic")
PROJECT_SRC = PROJECT_ROOT / "src"
if str(PROJECT_SRC) not in sys.path:
    sys.path.append(str(PROJECT_SRC))

from thematic_topic.config import load_config, PipelineConfig
from thematic_topic.headlines import fetch_headlines
from thematic_topic.preprocess import clean_headlines
from thematic_topic.embeddings import encode_texts_with_cache
from thematic_topic.dedup import deduplicate_headlines
from thematic_topic.topics import fit_topic_model, transform_topics
from thematic_topic.intensity import build_topic_intensity


def _select_fit_sample(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")

    fit_start = cfg.topic_model.fit_start
    fit_end = cfg.topic_model.fit_end
    if fit_start:
        out = out[out["timestamp"] >= pd.to_datetime(fit_start, utc=True)]
    if fit_end:
        out = out[out["timestamp"] <= pd.to_datetime(fit_end, utc=True)]

    if out.empty:
        raise ValueError("BERTopic学習期間に該当するニュースがありません。")
    return out.reset_index(drop=True)


cfg = load_config(path=PROJECT_ROOT / "config/default.yaml", project_root=PROJECT_ROOT)

ld.open_session()
try:
    # 1) LSEG取得 + 正規化
    raw_df, normalized_df = fetch_headlines(cfg)

    # 2) 前処理
    clean_df = clean_headlines(
        normalized_df,
        mode=cfg.preprocess.low_information_mode,
        min_chars=cfg.preprocess.min_headline_chars,
    )

    # 3) embedding（本文主入力: model_input_text）
    cache_path = cfg.resolve_path(cfg.embedding.headline_cache_path)
    emb_meta, all_embeddings = encode_texts_with_cache(
        ids=clean_df["news_id"],
        texts=clean_df["model_input_text"],
        cache_path=cache_path,
        model_name=cfg.embedding.model_name,
        batch_size=cfg.embedding.batch_size,
        normalize_embeddings=cfg.embedding.normalize_embeddings,
    )

    # 4) 重複抑制
    dedup_df, dedup_log = deduplicate_headlines(
        clean_df,
        embeddings=all_embeddings,
        similarity_threshold=cfg.dedup.similarity_threshold,
        window_hours=cfg.dedup.window_hours,
    )

    # 5) dedup後embedding再マッピング
    clean_pos = clean_df.reset_index()[["index", "news_id"]]
    dedup_indexer = (
        dedup_df[["news_id"]]
        .merge(clean_pos, on="news_id", how="left")["index"]
        .to_numpy(dtype=int)
    )
    dedup_emb = all_embeddings[dedup_indexer]

    # 6) BERTopic fit/transform
    fit_df = _select_fit_sample(dedup_df, cfg)
    dedup_pos = dedup_df.reset_index()[["index", "news_id"]]
    fit_idx = (
        fit_df[["news_id"]]
        .merge(dedup_pos, on="news_id", how="left")["index"]
        .to_numpy(dtype=int)
    )
    fit_emb = dedup_emb[fit_idx]

    topic_model, fit_tables = fit_topic_model(
        fit_df, fit_emb, cfg.topic_model, text_col="model_input_text"
    )
    topic_assignments = transform_topics(
        dedup_df, topic_model, dedup_emb, text_col="model_input_text"
    )

    # 7) topic強度（daily/weekly）
    daily_intensity, weekly_intensity, outlier_stats = build_topic_intensity(
        topic_assignments,
        weekly_rule=cfg.topic_intensity.weekly_rule,       # "W-FRI"
        ewma_span=cfg.topic_intensity.ewma_span,           # 5
        lookback=cfg.topic_intensity.lookback,             # 20
        z_threshold=cfg.topic_intensity.z_threshold,       # 1.0
        aggregate_timezone=cfg.topic_intensity.aggregate_timezone,  # "Asia/Tokyo"
    )
finally:
    ld.close_session()

print("weekly_intensity shape:", weekly_intensity.shape)
display(weekly_intensity.head(20))
display(outlier_stats)
